## 1. Create directories

In [19]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

SRC_DIR = Path("../src")
DATA_DIR = Path("../data/processed")
EMBEDDING_DIR = Path("../data/embeddings")

SRC_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Directories ready")


✅ Directories ready


## 2. Create `src/data_loader.py`

In [20]:
data_loader_code = '\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\n\nDEFAULT_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "multi_city_recommendation_features.csv"\nDEFAULT_EMBEDDING_PATH = PROJECT_ROOT / "data" / "embeddings" / "multi_city_place_embeddings.npy"\n\ndef load_places(data_path=DEFAULT_DATA_PATH):\n    path = Path(data_path)\n    if not path.exists():\n        raise FileNotFoundError(f"Place dataset not found: {path}")\n    return pd.read_csv(path)\n\ndef load_embeddings(embedding_path=DEFAULT_EMBEDDING_PATH):\n    path = Path(embedding_path)\n    if not path.exists():\n        raise FileNotFoundError(f"Embedding file not found: {path}")\n    return np.load(path)\n\ndef available_cities(df):\n    if "city" not in df.columns:\n        raise KeyError("Expected a city column.")\n    return sorted(df["city"].dropna().astype(str).str.strip().unique().tolist())\n'
(SRC_DIR / 'data_loader.py').write_text(data_loader_code, encoding='utf-8')
print('✅ src/data_loader.py created')

✅ src/data_loader.py created


## 3. Create `src/recommender.py`

In [21]:
recommender_code = '\nimport numpy as np\nimport pandas as pd\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics.pairwise import cosine_similarity\nfrom sentence_transformers import SentenceTransformer\n\nFEATURE_COLUMNS = [\n    "nature", "history", "culture", "adventure",\n    "photography", "shopping", "religious", "family"\n]\n\nclass TravelRecommender:\n    # City-aware hybrid recommender.\n\n    def __init__(self, df, place_embeddings=None, model_name="sentence-transformers/all-MiniLM-L6-v2"):\n        self.df = df.copy()\n\n        required = {"city", "name", "category", "travel_tags", "rating", "reviews"}\n        missing = sorted(required - set(self.df.columns))\n        if missing:\n            raise ValueError(f"Missing recommender columns: {missing}")\n\n        for col in ["city", "name", "category", "travel_tags"]:\n            self.df[col] = self.df[col].fillna("").astype(str).str.strip()\n\n        self.df["rating_score"] = self._minmax(self.df["rating"])\n        self.df["popularity_score"] = self._minmax(\n            np.log1p(\n                pd.to_numeric(\n                    self.df["reviews"],\n                    errors="coerce"\n                ).clip(lower=0)\n            )\n        )\n\n        for feature in FEATURE_COLUMNS:\n            if feature not in self.df.columns:\n                self.df[feature] = 0\n\n        self.structured_matrix = (\n            self.df[FEATURE_COLUMNS]\n            .fillna(0)\n            .astype(float)\n            .to_numpy()\n        )\n\n        self.df["recommendation_text"] = (\n            self.df["name"] + ". "\n            + self.df["city"] + ". "\n            + self.df["category"] + ". "\n            + self.df["travel_tags"]\n        ).str.lower()\n\n        self.tfidf_vectorizer = TfidfVectorizer(\n            stop_words="english",\n            ngram_range=(1, 2)\n        )\n        self.tfidf_matrix = self.tfidf_vectorizer.fit_transform(\n            self.df["recommendation_text"]\n        )\n\n        self.semantic_model = SentenceTransformer(model_name)\n\n        if place_embeddings is None:\n            self.place_embeddings = self.semantic_model.encode(\n                self.df["recommendation_text"].tolist(),\n                normalize_embeddings=True,\n                show_progress_bar=True\n            )\n        else:\n            if len(place_embeddings) != len(self.df):\n                raise ValueError(\n                    "Embedding row count does not match dataset row count."\n                )\n            self.place_embeddings = place_embeddings\n\n    @staticmethod\n    def _minmax(series):\n        values = pd.to_numeric(series, errors="coerce").fillna(0.0)\n        lo, hi = values.min(), values.max()\n\n        if lo == hi:\n            return pd.Series(np.ones(len(values)), index=values.index)\n\n        return (values - lo) / (hi - lo)\n\n    def recommend(self, destination, query, user_preferences, top_n=5):\n        missing = [\n            f for f in FEATURE_COLUMNS\n            if f not in user_preferences\n        ]\n        if missing:\n            raise ValueError(\n                f"Missing preference fields: {missing}"\n            )\n\n        values = [\n            float(user_preferences[f])\n            for f in FEATURE_COLUMNS\n        ]\n\n        if any(v < 0 or v > 1 for v in values):\n            raise ValueError(\n                "Preference values must be between 0 and 1."\n            )\n\n        city = destination.strip().lower()\n\n        positions = np.flatnonzero(\n            (self.df["city"].str.lower() == city).to_numpy()\n        )\n\n        if len(positions) == 0:\n            raise ValueError(\n                f"Destination \'{destination}\' not found. "\n                f"Available: {sorted(self.df[\'city\'].unique())}"\n            )\n\n        user_vector = np.array(values).reshape(1, -1)\n\n        structured_scores = cosine_similarity(\n            user_vector,\n            self.structured_matrix[positions]\n        ).flatten()\n\n        query_text = f"{destination}. {query}"\n\n        tfidf_query = self.tfidf_vectorizer.transform(\n            [query_text.lower()]\n        )\n\n        tfidf_scores = cosine_similarity(\n            tfidf_query,\n            self.tfidf_matrix[positions]\n        ).flatten()\n\n        query_embedding = self.semantic_model.encode(\n            [query_text],\n            normalize_embeddings=True\n        )\n\n        semantic_scores = cosine_similarity(\n            query_embedding,\n            self.place_embeddings[positions]\n        ).flatten()\n\n        result = (\n            self.df.iloc[positions]\n            .copy()\n            .reset_index(drop=True)\n        )\n\n        result["structured_score"] = structured_scores\n        result["tfidf_score"] = tfidf_scores\n        result["semantic_score"] = semantic_scores\n\n        result["final_score"] = (\n            0.25 * result["structured_score"]\n            + 0.25 * result["tfidf_score"]\n            + 0.30 * result["semantic_score"]\n            + 0.10 * result["rating_score"]\n            + 0.10 * result["popularity_score"]\n        )\n\n        return (\n            result\n            .sort_values("final_score", ascending=False)\n            .head(top_n)\n            .reset_index(drop=True)\n        )\n'
(SRC_DIR / 'recommender.py').write_text(recommender_code, encoding='utf-8')
print('✅ src/recommender.py created')

✅ src/recommender.py created


## 4. Create `src/itinerary.py`

In [22]:
itinerary_code = '\nimport math\nimport numpy as np\nimport pandas as pd\n\nDEFAULT_SPEED_KMPH = 25.0\nDEFAULT_DAY_START_MINUTES = 9 * 60\nDEFAULT_MAX_DAY_MINUTES = 7 * 60\n\ndef haversine_km(lat1, lon1, lat2, lon2):\n    R = 6371.0\n    lat1, lon1 = math.radians(lat1), math.radians(lon1)\n    lat2, lon2 = math.radians(lat2), math.radians(lon2)\n    dlat, dlon = lat2 - lat1, lon2 - lon1\n\n    a = (\n        math.sin(dlat / 2) ** 2\n        + math.cos(lat1) * math.cos(lat2)\n        * math.sin(dlon / 2) ** 2\n    )\n\n    return 2 * R * math.atan2(\n        math.sqrt(a),\n        math.sqrt(1 - a)\n    )\n\ndef format_time(minutes):\n    minutes = int(round(minutes))\n    hour = (minutes // 60) % 24\n    minute = minutes % 60\n    suffix = "AM" if hour < 12 else "PM"\n    display_hour = hour % 12 or 12\n    return f"{display_hour}:{minute:02d} {suffix}"\n\ndef build_travel_time_matrix(\n    candidates,\n    average_speed_kmph=DEFAULT_SPEED_KMPH\n):\n    if average_speed_kmph <= 0:\n        raise ValueError("average_speed_kmph must be > 0.")\n\n    n = len(candidates)\n    matrix = np.zeros((n, n), dtype=float)\n\n    for i in range(n):\n        for j in range(n):\n            distance = haversine_km(\n                candidates.iloc[i]["latitude"],\n                candidates.iloc[i]["longitude"],\n                candidates.iloc[j]["latitude"],\n                candidates.iloc[j]["longitude"]\n            )\n            matrix[i, j] = distance / average_speed_kmph * 60\n\n    return matrix\n\ndef generate_itinerary(\n    candidates,\n    days=3,\n    max_day_minutes=DEFAULT_MAX_DAY_MINUTES,\n    average_speed_kmph=DEFAULT_SPEED_KMPH,\n    day_start_minutes=DEFAULT_DAY_START_MINUTES\n):\n    required = {\n        "city", "name", "latitude", "longitude",\n        "activity_type", "estimated_visit_minutes",\n        "estimated_price_level", "final_score"\n    }\n\n    missing = sorted(required - set(candidates.columns))\n    if missing:\n        raise ValueError(f"Missing itinerary columns: {missing}")\n\n    if days < 1 or max_day_minutes <= 0:\n        raise ValueError("days and max_day_minutes must be positive.")\n\n    candidates = candidates.copy().reset_index(drop=True)\n\n    if candidates.empty:\n        return pd.DataFrame()\n\n    matrix = build_travel_time_matrix(\n        candidates,\n        average_speed_kmph\n    )\n\n    remaining = set(range(len(candidates)))\n    rows = []\n\n    for day in range(1, days + 1):\n        if not remaining:\n            break\n\n        current = max(\n            remaining,\n            key=lambda idx: float(\n                candidates.iloc[idx]["final_score"]\n            )\n        )\n\n        used = 0.0\n        stop = 1\n\n        while remaining:\n            feasible = []\n\n            for idx in remaining:\n                travel = (\n                    0.0\n                    if used == 0\n                    else matrix[current, idx]\n                )\n\n                visit = float(\n                    candidates.iloc[idx]["estimated_visit_minutes"]\n                )\n\n                total = used + travel + visit\n\n                if total <= max_day_minutes + 1e-9:\n                    score = float(\n                        candidates.iloc[idx]["final_score"]\n                    )\n                    efficiency = score / (1.0 + travel)\n                    feasible.append(\n                        (idx, travel, visit, efficiency)\n                    )\n\n            if not feasible:\n                break\n\n            idx, travel, visit, _ = max(\n                feasible,\n                key=lambda item: item[3]\n            )\n\n            arrival = day_start_minutes + used + travel\n            departure = arrival + visit\n\n            rows.append({\n                "city": candidates.iloc[idx]["city"],\n                "day": day,\n                "stop": stop,\n                "place": candidates.iloc[idx]["name"],\n                "activity_type": candidates.iloc[idx]["activity_type"],\n                "arrival": format_time(arrival),\n                "departure": format_time(departure),\n                "travel_before_minutes": round(travel, 2),\n                "visit_minutes": int(round(visit)),\n                "estimated_price_level": int(\n                    round(candidates.iloc[idx]["estimated_price_level"])\n                ),\n                "final_score": round(\n                    float(candidates.iloc[idx]["final_score"]),\n                    4\n                ),\n                "latitude": float(candidates.iloc[idx]["latitude"]),\n                "longitude": float(candidates.iloc[idx]["longitude"]),\n            })\n\n            used += travel + visit\n            current = idx\n            remaining.remove(idx)\n            stop += 1\n\n    result = pd.DataFrame(rows)\n\n    if not result.empty:\n        totals = (\n            result\n            .assign(\n                total_minutes=lambda x:\n                    x["travel_before_minutes"] + x["visit_minutes"]\n            )\n            .groupby(["city", "day"])["total_minutes"]\n            .sum()\n        )\n\n        if (totals > max_day_minutes + 1e-9).any():\n            raise RuntimeError(\n                "Generated itinerary violates daily budget."\n            )\n\n    return result\n'
(SRC_DIR / 'itinerary.py').write_text(itinerary_code, encoding='utf-8')
print('✅ src/itinerary.py created')

✅ src/itinerary.py created


## 5. Create `src/utils.py` and package initializer

In [ ]:
utils_code = '\ndef validate_city_output(result, requested_city):\n    if result.empty:\n        return False\n\n    return (\n        result["city"]\n        .astype(str)\n        .str.strip()\n        .str.lower()\n        .eq(requested_city.strip().lower())\n        .all()\n    )\n\ndef validate_daily_budget(result, max_day_minutes):\n    if result.empty:\n        return True\n\n    totals = (\n        result\n        .assign(\n            total_minutes=lambda x:\n                x["travel_before_minutes"] + x["visit_minutes"]\n        )\n        .groupby(["city", "day"])["total_minutes"]\n        .sum()\n    )\n\n    return bool(\n        (totals <= max_day_minutes + 1e-9).all()\n    )\n'
(SRC_DIR / 'utils.py').write_text(utils_code, encoding='utf-8')
(SRC_DIR / '__init__.py').write_text('', encoding='utf-8')
print('✅ src/utils.py and src/__init__.py created')

✅ src/utils.py and src/__init__.py created


## 6. Import the shared engine

In [27]:
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_places, available_cities
from src.recommender import TravelRecommender
from src.itinerary import generate_itinerary
from src.utils import validate_city_output, validate_daily_budget

print("✅ Shared engine imported")


✅ Shared engine imported


## 7. Load data and build the full multi-city recommender

In [28]:
places = load_places()

print("Dataset:", places.shape)
print("Cities:", available_cities(places))

recommender = TravelRecommender(
    df=places,
    place_embeddings=None
)

print(
    "Embedding shape:",
    recommender.place_embeddings.shape
)

np.save(
    EMBEDDING_DIR / "multi_city_place_embeddings.npy",
    recommender.place_embeddings
)

print("✅ Full multi-city embeddings saved")


Dataset: (120, 27)
Cities: ['Goa', 'Jaipur', 'Manali', 'Rishikesh', 'Shimla', 'Udaipur']


Batches: 100%|██████████| 4/4 [00:00<00:00,  9.89it/s]

Embedding shape: (120, 384)
✅ Full multi-city embeddings saved


## 8. Test Goa recommendation

In [29]:
goa_preferences = {
    "nature": 1.0,
    "history": 0.1,
    "culture": 0.4,
    "adventure": 0.4,
    "photography": 1.0,
    "shopping": 0.2,
    "religious": 0.0,
    "family": 0.5
}

goa = recommender.recommend(
    "Goa",
    "beautiful beaches and relaxed scenic places",
    goa_preferences,
    top_n=5
)

display(
    goa[[
        "name",
        "city",
        "rating",
        "semantic_score",
        "final_score"
    ]]
)

assert validate_city_output(goa, "Goa")
print("✅ Goa recommendation test passed")


,name,city,rating,semantic_score,final_score
0,"Calangute Beach, Goa",Goa,4.3,0.830518,0.718666
1,Keri Beach,Goa,4.6,0.816173,0.662736
2,Kuske Waterfall,Goa,4.4,0.731620,0.593910
3,Dudhsagar Falls,Goa,4.6,0.699319,0.584445
4,Cabo de Rama Fort South Goa,Goa,4.4,0.729781,0.568416


✅ Goa recommendation test passed


## 9. Test complete Goa recommendation → itinerary flow

In [30]:
goa_candidates = recommender.recommend(
    "Goa",
    "beautiful beaches and relaxed scenic places",
    goa_preferences,
    top_n=12
)

goa_candidates = goa_candidates.copy()
goa_candidates["activity_type"] = "sightseeing"
goa_candidates["estimated_visit_minutes"] = 60
goa_candidates["estimated_price_level"] = 1

goa_itinerary = generate_itinerary(
    goa_candidates,
    days=3
)

display(goa_itinerary)

assert validate_city_output(
    goa_itinerary,
    "Goa"
)

assert validate_daily_budget(
    goa_itinerary,
    7 * 60
)

print("✅ Goa itinerary test passed")


,city,day,stop,place,activity_type,arrival,departure,travel_before_minutes,visit_minutes,estimated_price_level,final_score,latitude,longitude
0,Goa,1,1,"Calangute Beach, Goa",sightseeing,9:00 AM,10:00 AM,0.00,60,1,0.7187,15.544721,73.754669
1,Goa,1,2,Sinquerim Fort,sightseeing,10:13 AM,11:13 AM,12.71,60,1,0.5581,15.498466,73.766404
2,Goa,1,3,Fort Aguada,sightseeing,11:15 AM,12:15 PM,2.51,60,1,0.5412,15.492252,73.773746
3,Goa,1,4,Chapora Fort,sightseeing,12:47 PM,1:47 PM,31.45,60,1,0.4944,15.604638,73.736963
4,Goa,1,5,Keri Beach,sightseeing,2:17 PM,3:17 PM,30.00,60,1,0.6627,15.708774,73.692984
5,Goa,2,1,Kuske Waterfall,sightseeing,9:00 AM,10:00 AM,0.00,60,1,0.5939,15.021708,74.208660
6,Goa,2,2,Cabo de Rama Fort South Goa,sightseeing,11:16 AM,12:16 PM,76.11,60,1,0.5684,15.088785,73.921593
7,Goa,2,3,Sunset View Point Colva,sightseeing,1:06 PM,2:06 PM,49.70,60,1,0.5292,15.274854,73.913585
8,Goa,2,4,Goa Chitra Museum,sightseeing,2:13 PM,3:13 PM,7.62,60,1,0.4894,15.264899,73.941336
9,Goa,3,1,Dudhsagar Falls,sightseeing,9:00 AM,10:00 AM,0.00,60,1,0.5844,15.314438,74.314307


✅ Goa itinerary test passed


## 10. Test every supported city

In [31]:
test_preferences = {
    "nature": 0.7,
    "history": 0.3,
    "culture": 0.4,
    "adventure": 0.4,
    "photography": 0.7,
    "shopping": 0.3,
    "religious": 0.2,
    "family": 0.4
}

test_rows = []
city_itineraries = {}

for city in available_cities(places):

    recommendations = recommender.recommend(
        city,
        "best places for a memorable trip",
        test_preferences,
        top_n=12
    )

    recommendations = recommendations.copy()
    recommendations["activity_type"] = "sightseeing"
    recommendations["estimated_visit_minutes"] = 60
    recommendations["estimated_price_level"] = 1

    plan = generate_itinerary(
        recommendations,
        days=3
    )

    city_itineraries[city] = plan

    test_rows.append({
        "city": city,
        "recommendations": len(recommendations),
        "itinerary_stops": len(plan),
        "city_valid": validate_city_output(plan, city),
        "budget_valid": validate_daily_budget(plan, 7 * 60)
    })

engine_tests = pd.DataFrame(test_rows)
engine_tests


,city,recommendations,itinerary_stops,city_valid,budget_valid
0,Goa,12,12,True,True
1,Jaipur,12,12,True,True
2,Manali,12,12,True,True
3,Rishikesh,12,12,True,True
4,Shimla,12,12,True,True
5,Udaipur,12,12,True,True


In [32]:
assert engine_tests["city_valid"].all()
assert engine_tests["budget_valid"].all()

print("✅ Shared engine passed all city and budget tests.")


✅ Shared engine passed all city and budget tests.


## 11. Save the shared-engine itinerary output

In [33]:
all_plans = pd.concat(
    [
        plan
        for plan in city_itineraries.values()
        if not plan.empty
    ],
    ignore_index=True
)

output_path = (
    DATA_DIR
    / "multi_city_itineraries_shared_engine.csv"
)

all_plans.to_csv(
    output_path,
    index=False
)

print(f"✅ Saved: {output_path}")


✅ Saved: ..\data\processed\multi_city_itineraries_shared_engine.csv


# 🎯 Architecture milestone

Reusable intelligence now lives in:

```text
src/
├── data_loader.py
├── recommender.py
├── itinerary.py
├── utils.py
└── __init__.py
```

Both FastAPI and Streamlit can call these same modules.

## Next

`18_fastapi_multicity.ipynb`

We will replace the duplicated backend logic with the shared engine and make both `/recommend` and `/itinerary` work with all supported destinations.
